In [19]:
from winnow_fcns import *
#first to siffting
#run error orrectin protocal on that
#do naive agreement

In [20]:
# 7 bit case
# 7 word blocks, calculate the syndrome, compare the two
# send the syndrome over public channel, you'll lose 3 bits
# run a big loop 
# groups of 7 bits, calculate syndrome xor with the data to get the syndrome
# increase number of bits for blocks to see whats optimal
# start with as large bits as you can 
# hamming code can correct one error and detect 2
# logic for if its better to error detect and throw some data away vs error correct
# check to see if you still have errors, must scramble the data to ensure ur error correction works bcs errrors tend to be grouped together
# keep track of how many bits you expose
# detune system tio change qubit error rate of physical system

In [21]:
def simulate_error(key, rng, ber=0.25, N=10):
    mask = (rng.random(size=N) < ber).astype(int)
    
    cooked_key = key ^ mask
    
    return cooked_key

rng = np.random.default_rng(seed=42)
rng2 = np.random.default_rng(seed=180)
N = 30
std = 0.1
ber = 0.01


init_key = rng2.integers(low=0, high=2, size=N)

print(init_key)
key = simulate_error(init_key, rng, ber=ber, N=N)
print(key)

[0 1 1 1 0 1 1 0 0 0 1 1 0 0 1 1 0 1 1 0 0 1 0 0 0 0 0 1 1 1]
[0 1 1 1 0 1 1 0 0 0 1 1 0 0 1 1 0 1 1 0 0 1 0 0 0 0 0 1 1 1]


In [22]:
import numpy as np

# create a mock BitBuffer class for testing
class MockBitBuffer:
    def __init__(self, bits):
        self.bits = list(bits)
        self.seed = None
    def get_length(self): return len(self.bits)
    def get_bit(self, i): return self.bits[i]
    def set_bit(self, i): self.bits[i] = 1
    def clear_bit(self, i): self.bits[i] = 0
    def flip_bit(self, i): self.bits[i] = 1 - self.bits[i]
    def set_seed(self, s): self.seed = s
    def permute_buffer(self): pass # Simplified for test

#create two keys that differ by exactly one bit in the first block
alice_key = MockBitBuffer([1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0])
bob_key   = MockBitBuffer([1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0]) # index 1 is different

#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

# alice calculates syndrome for the first block 
alice_syndrome = alice_winnow.get_syndrome(0)

# bob uses Alice's syndrome to fix his key
print(f"Bob's key before fix: {bob_key.bits[:8]}")
bob_winnow.fix_with_syndrome(0, alice_syndrome)
print(f"Bob's key after fix:  {bob_key.bits[:8]}")

# verify that the keys match
if alice_key.bits == bob_key.bits:
    print(" Success! Bob corrected the error.")
else:
    print("Failure: Keys still do not match.")

Bob's key before fix: [1, 1, 1, 1, 0, 0, 1, 0]
Bob's key after fix:  [1, 0, 1, 1, 0, 0, 1, 0]
 Success! Bob corrected the error.


In [23]:
alice_key = MockBitBuffer(init_key)
bob_key   = MockBitBuffer(key)


#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

# alice calculates syndrome for the first block 
alice_syndrome = alice_winnow.get_syndrome(0)

# bob uses Alice's syndrome to fix his key
print(f"Bob's key before fix: {bob_key.bits[:8]}")
bob_winnow.fix_with_syndrome(0, alice_syndrome)
print(f"Bob's key after fix:  {bob_key.bits[:8]}")
print(f"Alice's key: {alice_key.bits[:8]}")

# verify that the keys match
if alice_key.bits == bob_key.bits:
    print(" Success! Bob corrected the error.")
else:
    print("Failure: Keys still do not match.")

Bob's key before fix: [np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0)]
Bob's key after fix:  [np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0)]
Alice's key: [np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0)]
 Success! Bob corrected the error.
